In [11]:
import pandas as pd

# 1. Load the master dataset
df = pd.read_csv("acs_demographics.csv")

In [12]:
# 2. Calculate Demographic Percentages
# Using Total Population (B01003_001E) as the denominator
df['pct_white'] = (df['B03002_003E'] / df['B01003_001E']) * 100
df['pct_black'] = (df['B03002_004E'] / df['B01003_001E']) * 100
df['pct_asian'] = (df['B03002_006E'] / df['B01003_001E']) * 100
df['pct_hispanic'] = (df['B03002_012E'] / df['B01003_001E']) * 100

# 3. Calculate Infrastructure & Socioeconomic Percentages
# Transit: Public transit commuters (B08301_010E) / Total Workers 16+ (B08301_001E)
df['pct_transit_commuters'] = (df['B08301_010E'] / df['B08301_001E']) * 100

# Broadband: Broadband households (B28002_004E) / Total Households (B28002_001E)
df['pct_broadband_access'] = (df['B28002_004E'] / df['B28002_001E']) * 100

# Rent Burden: Severely rent-burdened (B25070_010E) / Total Households (B28002_001E)
df['pct_severe_rent_burden'] = (df['B25070_010E'] / df['B28002_001E']) * 100

# 4. Rename the remaining raw columns for readability
df.rename(columns={
    'B19013_001E': 'median_income',
    'B01003_001E': 'total_population'
}, inplace=True)

# 5. Filter the dataframe to only the columns we care about and round to 2 decimals
analysis_cols = [
    'city_name', 'total_population', 'median_income', 
    'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic',
    'pct_transit_commuters', 'pct_broadband_access', 'pct_severe_rent_burden'
]
analysis_df = df[analysis_cols].round(2)

# Save this cleaned data for the NLP portion
analysis_df.to_csv("cleaned_demographic_metrics.csv", index=False)

# 6. Print a comparison against the County Baseline
print("--- DEMOGRAPHIC ANALYSIS SUMMARY ---")
print(analysis_df[['city_name', 'total_population', 'median_income', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic', 'pct_severe_rent_burden', 'pct_transit_commuters', 'pct_broadband_access',]].to_string(index=False))

print("\n--- BASELINE COMPARISON ---")
# Extract the county baseline row
county_baseline = analysis_df[analysis_df['city_name'] == 'Orange County (Baseline)'].iloc[0]

# Calculate how each city deviates from the county average for Rent Burden
for index, row in analysis_df.iterrows():
    if row['city_name'] != 'Orange County (Baseline)':
        diff = row['pct_severe_rent_burden'] - county_baseline['pct_severe_rent_burden']
        direction = "higher" if diff > 0 else "lower"
        print(f"{row['city_name']}: Severe rent burden is {abs(diff):.1f}% {direction} than the county average.")

--- DEMOGRAPHIC ANALYSIS SUMMARY ---
               city_name  total_population  median_income  pct_white  pct_black  pct_asian  pct_hispanic  pct_severe_rent_burden  pct_transit_commuters  pct_broadband_access
Orange County (Baseline)            145919          85785      68.50      10.64       7.79          8.73                   10.63                   4.73                 92.44
             Chapel Hill             58919          85940      64.65      10.32      13.38          6.77                   18.50                   7.03                 92.45
                Carrboro             21242          76933      65.10      12.87       9.05          8.06                   13.73                   9.38                 96.05
            Hillsborough              9534          86250      73.81      12.43       2.34          7.99                    8.07                   1.55                 94.06
                  Mebane             17899          78419      57.12      24.41       4.42   